In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import torch
import tensorflow as tf


In [ ]:
base_path = "/Users/amirhossein/Projects/NJIT/Data/MeshGraphNets/datasets/cylinder_flow"

In [ ]:
tfrecord_path = os.path.join(base_path, "test.tfrecord")

In [ ]:
# Load meta.json
meta_path = Path(base_path) / "meta.json"
with open(meta_path, 'r') as f:
    meta = json.load(f)

print("=" * 50)
print("META.JSON CONTENT:")
print("=" * 50)
print(json.dumps(meta, indent=2))

In [ ]:
# Correct parser for MeshGraphNets TFRecord format
def parse_meshgraphnets_tfrecord(example_proto):
    """Parse MeshGraphNets TFRecord - data is stored as serialized bytes."""

    feature_description = {
        'cells': tf.io.FixedLenFeature([], tf.string),
        'mesh_pos': tf.io.FixedLenFeature([], tf.string),
        'node_type': tf.io.FixedLenFeature([], tf.string),
        'velocity': tf.io.FixedLenFeature([], tf.string),
        'pressure': tf.io.FixedLenFeature([], tf.string),
    }

    parsed = tf.io.parse_single_example(example_proto, feature_description)

    # Decode serialized tensors
    cells = tf.io.decode_raw(parsed['cells'], tf.int32)
    mesh_pos = tf.io.decode_raw(parsed['mesh_pos'], tf.float32)
    node_type = tf.io.decode_raw(parsed['node_type'], tf.int32)
    velocity = tf.io.decode_raw(parsed['velocity'], tf.float32)
    pressure = tf.io.decode_raw(parsed['pressure'], tf.float32)

    # Convert to numpy then PyTorch
    return {
        'cells': torch.from_numpy(cells.numpy()),
        'mesh_pos': torch.from_numpy(mesh_pos.numpy()),
        'node_type': torch.from_numpy(node_type.numpy()),
        'velocity': torch.from_numpy(velocity.numpy()),
        'pressure': torch.from_numpy(pressure.numpy()),
    }


# Test parsing
tfrecord_path = Path(base_path) / "test.tfrecord"
dataset = tf.data.TFRecordDataset(str(tfrecord_path))

print("\n" + "=" * 50)
print("PARSED DATA (Raw):")
print("=" * 50)

for i, raw_record in enumerate(dataset.take(1)):
    parsed = parse_meshgraphnets_tfrecord(raw_record)

    for key, value in parsed.items():
        print(f"\n{key}:")
        print(f"  Total elements: {value.shape[0]}")
        print(f"  Dtype: {value.dtype}")
        print(f"  First 10 values: {value[:10]}")

In [ ]:
# Now reshape the data based on the structure
def parse_and_reshape_meshgraphnets(example_proto, meta):
    """Parse and reshape MeshGraphNets data to proper dimensions."""

    feature_description = {
        'cells': tf.io.FixedLenFeature([], tf.string),
        'mesh_pos': tf.io.FixedLenFeature([], tf.string),
        'node_type': tf.io.FixedLenFeature([], tf.string),
        'velocity': tf.io.FixedLenFeature([], tf.string),
        'pressure': tf.io.FixedLenFeature([], tf.string),
    }

    parsed = tf.io.parse_single_example(example_proto, feature_description)

    # Decode serialized tensors
    cells_raw = tf.io.decode_raw(parsed['cells'], tf.int32).numpy()
    mesh_pos_raw = tf.io.decode_raw(parsed['mesh_pos'], tf.float32).numpy()
    node_type_raw = tf.io.decode_raw(parsed['node_type'], tf.int32).numpy()
    velocity_raw = tf.io.decode_raw(parsed['velocity'], tf.float32).numpy()
    pressure_raw = tf.io.decode_raw(parsed['pressure'], tf.float32).numpy()

    # Reshape based on expected dimensions
    # cells: (num_cells, 3) - triangles
    num_cells = len(cells_raw) // 3
    cells = cells_raw.reshape(num_cells, 3)

    # mesh_pos: (num_nodes, 2) - (x, y) coordinates
    num_nodes = len(mesh_pos_raw) // 2
    mesh_pos = mesh_pos_raw.reshape(num_nodes, 2)

    # node_type: (num_nodes,)
    node_type = node_type_raw  # Already 1D

    # velocity: (timesteps, num_nodes, 2) - (vx, vy) at each node over time
    trajectory_length = meta.get('trajectory_length', 600)
    velocity = velocity_raw.reshape(trajectory_length, num_nodes, 2)

    # pressure: (timesteps, num_nodes)
    pressure = pressure_raw.reshape(trajectory_length, num_nodes)

    # Convert to PyTorch tensors
    return {
        'cells': torch.from_numpy(cells).long(),
        'mesh_pos': torch.from_numpy(mesh_pos).float(),
        'node_type': torch.from_numpy(node_type).long(),
        'velocity': torch.from_numpy(velocity).float(),
        'pressure': torch.from_numpy(pressure).float(),
    }


# Parse and display structure
print("\n" + "=" * 50)
print("MESHGRAPHNETS DATA STRUCTURE (PyTorch):")
print("=" * 50)

for i, raw_record in enumerate(dataset.take(1)):
    parsed = parse_and_reshape_meshgraphnets(raw_record, meta)

    print(f"\nTrajectory {i}:")
    print("-" * 50)

    for key, value in parsed.items():
        print(f"\n{key}:")
        print(f"  Shape: {value.shape}")
        print(f"  Dtype: {value.dtype}")
        print(f"  Device: {value.device}")
        print(f"  Min: {value.min().item():.6f}")
        print(f"  Max: {value.max().item():.6f}")

        if key == 'cells':
            print(f"  Description: Triangular mesh connectivity (num_cells={value.shape[0]})")
            print(f"  First 3 triangles:\n{value[:3]}")
        elif key == 'mesh_pos':
            print(f"  Description: Node positions (num_nodes={value.shape[0]})")
            print(f"  First 3 positions:\n{value[:3]}")
        elif key == 'node_type':
            unique_types = torch.unique(value)
            print(f"  Description: Node type markers")
            print(f"  Unique types: {unique_types.tolist()}")
            for nt in unique_types:
                count = (value == nt).sum().item()
                print(f"    Type {nt}: {count} nodes")
        elif key == 'velocity':
            print(f"  Description: Velocity field over time (timesteps={value.shape[0]}, nodes={value.shape[1]})")
            print(f"  First timestep, first 3 nodes:\n{value[0, :3]}")
        elif key == 'pressure':
            print(f"  Description: Pressure field over time (timesteps={value.shape[0]}, nodes={value.shape[1]})")
            print(f"  First timestep, first 3 nodes: {value[0, :3]}")

In [ ]:
# Load all trajectories
def load_all_trajectories(tfrecord_path, meta):
    """Load all trajectories from TFRecord file."""
    dataset = tf.data.TFRecordDataset(str(tfrecord_path))
    trajectories = []

    print("Loading trajectories...")
    for i, raw_record in enumerate(dataset):
        trajectory = parse_and_reshape_meshgraphnets(raw_record, meta)
        trajectories.append(trajectory)
        if (i + 1) % 10 == 0:
            print(f"  Loaded {i + 1} trajectories...")

    print(f"Total trajectories loaded: {len(trajectories)}")
    return trajectories


# Load all data
all_trajectories = load_all_trajectories(tfrecord_path, meta)

print("\n" + "=" * 50)
print("DATA LOADING COMPLETE!")
print("=" * 50)
print(f"\nTotal trajectories: {len(all_trajectories)}")
print(f"\nEach trajectory contains:")
print(f"  - cells: Mesh connectivity")
print(f"  - mesh_pos: Node positions (static)")
print(f"  - node_type: Boundary condition markers (static)")
print(f"  - velocity: Time series of velocity vectors")
print(f"  - pressure: Time series of pressure values")

print(f"\nExample access:")
print(f"  trajectory_0 = all_trajectories[0]")
print(f"  mesh_positions = trajectory_0['mesh_pos']  # Shape: {all_trajectories[0]['mesh_pos'].shape}")
print(f"  velocity_timeseries = trajectory_0['velocity']  # Shape: {all_trajectories[0]['velocity'].shape}")
print(f"  pressure_timeseries = trajectory_0['pressure']  # Shape: {all_trajectories[0]['pressure'].shape}")

In [ ]:
len(all_trajectories)

In [ ]:
# Create a convenient loader function
def load_meshgraphnets_data(base_path, split='test'):
    """
    Load MeshGraphNets cylinder flow data.

    Args:
        base_path: Path to the dataset directory
        split: 'test', 'train', or 'valid'

    Returns:
        meta: Dictionary with metadata
        trajectories: List of dictionaries with PyTorch tensors
    """
    base_path = Path(base_path)

    # Load metadata
    with open(base_path / "meta.json", 'r') as f:
        meta = json.load(f)

    # Load trajectories
    tfrecord_path = base_path / f"{split}.tfrecord"
    dataset = tf.data.TFRecordDataset(str(tfrecord_path))
    trajectories = []

    for raw_record in dataset:
        trajectory = parse_and_reshape_meshgraphnets(raw_record, meta)
        trajectories.append(trajectory)

    return meta, trajectories


print("Convenience function 'load_meshgraphnets_data' is now available!")
print("\nUsage:")
print("  meta, trajectories = load_meshgraphnets_data(base_path, split='test')")

In [ ]:

# Data summary and statistics
print("=" * 70)
print("CYLINDER FLOW DATASET SUMMARY")
print("=" * 70)

print(f"\n📊 DATASET INFO:")
print(f"  Number of trajectories: {len(all_trajectories)}")
print(f"  Trajectory length: {all_trajectories[0]['velocity'].shape[0]} timesteps")
print(f"  Number of nodes: {all_trajectories[0]['mesh_pos'].shape[0]}")
print(f"  Number of cells: {all_trajectories[0]['cells'].shape[0]}")

print(f"\n🔢 DATA DIMENSIONS:")
trajectory_0 = all_trajectories[0]
for key, tensor in trajectory_0.items():
    print(f"  {key:15s}: {str(tensor.shape):30s} dtype={tensor.dtype}")

print(f"\n🏷️  NODE TYPES:")
unique_types = torch.unique(trajectory_0['node_type'])
for nt in unique_types:
    count = (trajectory_0['node_type'] == nt).sum().item()
    pct = 100 * count / len(trajectory_0['node_type'])
    print(f"  Type {nt}: {count:4d} nodes ({pct:.1f}%)")

print(f"\n📈 VALUE RANGES:")
print(f"  Mesh X: [{trajectory_0['mesh_pos'][:, 0].min():.3f}, {trajectory_0['mesh_pos'][:, 0].max():.3f}]")
print(f"  Mesh Y: [{trajectory_0['mesh_pos'][:, 1].min():.3f}, {trajectory_0['mesh_pos'][:, 1].max():.3f}]")
print(
    f"  Velocity magnitude: [{torch.norm(trajectory_0['velocity'], dim=-1).min():.6f}, {torch.norm(trajectory_0['velocity'], dim=-1).max():.6f}]")
print(f"  Pressure: [{trajectory_0['pressure'].min():.6f}, {trajectory_0['pressure'].max():.6f}]")

print(f"\n💾 MEMORY USAGE:")
total_bytes = sum(t.element_size() * t.nelement() for traj in all_trajectories for t in traj.values())
print(f"  Total: {total_bytes / 1024 ** 2:.2f} MB")
print(f"  Per trajectory: {total_bytes / len(all_trajectories) / 1024 ** 2:.2f} MB")

print("\n" + "=" * 70)

In [ ]:

print("\n" + "🔷" * 35)
print("2️⃣  NODE TYPES - BOUNDARY CONDITIONS (CORRECTED)")
print("🔷" * 35)

trajectory = all_trajectories[0]  # First trajectory
node_type = trajectory['node_type']
mesh_pos = trajectory['mesh_pos']

print(f"""
Node types encode BOUNDARY CONDITIONS for the fluid simulation:
""")

# Analyze node types by position
for nt in torch.unique(node_type):
    mask = node_type == nt
    count = mask.sum().item()
    nodes_pos = mesh_pos[mask]

    x_min, x_max = nodes_pos[:, 0].min().item(), nodes_pos[:, 0].max().item()
    y_min, y_max = nodes_pos[:, 1].min().item(), nodes_pos[:, 1].max().item()

    print(f"\n  Type {nt}: {count} nodes")
    print(f"    X range: [{x_min:.4f}, {x_max:.4f}]")
    print(f"    Y range: [{y_min:.4f}, {y_max:.4f}]")

    # Corrected inference based on position
    if nt == 0:
        print(f"    → NORMAL (interior fluid nodes)")
    elif nt == 4:
        # Type 4: x=0, spanning Y direction
        print(f"    → INFLOW (left boundary, x = 0)")
    elif nt == 5:
        # Type 5: x=1.6, spanning Y direction
        print(f"    → OUTFLOW (right boundary, x = 1.6)")
    elif nt == 6:
        # Type 6: spans full X and Y range - must be walls + obstacle
        print(f"    → WALL/OBSTACLE (top/bottom walls + cylinder surface)")

print(f"""
Corrected node type meanings:
  • Type 0: NORMAL - interior fluid nodes (free to evolve)
  • Type 4: INFLOW - left boundary (x = 0), fixed inlet velocity
  • Type 5: OUTFLOW - right boundary (x = 1.6), open boundary
  • Type 6: WALL/OBSTACLE - top/bottom walls + cylinder (no-slip, v=0)

Type 6 includes:
  - Top wall (y ≈ 0.41)
  - Bottom wall (y ≈ 0)
  - Cylinder surface (center of domain)
  All have no-slip boundary condition (velocity = 0)
""")

In [ ]:

# Let's verify by checking which Type 6 nodes are walls vs cylinder
print("\n" + "🔷" * 35)
print("ANALYZING TYPE 6 NODES IN DETAIL")
print("🔷" * 35)

type6_mask = node_type == 6
type6_pos = mesh_pos[type6_mask]

# Separate by Y coordinate
bottom_wall_mask = type6_pos[:, 1] < 0.01
top_wall_mask = type6_pos[:, 1] > 0.40
cylinder_mask = ~(bottom_wall_mask | top_wall_mask)

print(f"\nType 6 breakdown:")
print(f"  Bottom wall (y ≈ 0):     {bottom_wall_mask.sum().item()} nodes")
print(f"  Top wall (y ≈ 0.41):     {top_wall_mask.sum().item()} nodes")
print(f"  Cylinder surface:        {cylinder_mask.sum().item()} nodes")
print(f"  Total Type 6:            {len(type6_pos)} nodes")

# Cylinder center and radius (approximate)
if cylinder_mask.sum() > 0:
    cylinder_nodes = type6_pos[cylinder_mask]
    center_x = cylinder_nodes[:, 0].mean().item()
    center_y = cylinder_nodes[:, 1].mean().item()
    radius = ((cylinder_nodes - cylinder_nodes.mean(dim=0)) ** 2).sum(dim=1).sqrt().max().item()

    print(f"\nCylinder properties:")
    print(f"  Approximate center: ({center_x:.4f}, {center_y:.4f})")
    print(f"  Approximate radius: {radius:.4f}")

In [ ]:

# Visualize the corrected understanding
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))

# Plot each node type with correct labels
colors = {0: 'lightblue', 4: 'green', 5: 'orange', 6: 'red'}
labels = {
    0: 'Type 0: Normal (interior fluid)',
    4: 'Type 4: Inflow (left, x=0)',
    5: 'Type 5: Outflow (right, x=1.6)',
    6: 'Type 6: Wall/Obstacle (no-slip)'
}

for nt in torch.unique(node_type):
    mask = node_type == nt
    ax.scatter(mesh_pos[mask, 0], mesh_pos[mask, 1],
               c=colors[nt.item()], s=30, label=labels[nt.item()],
               alpha=0.8, edgecolors='black', linewidth=0.5)

# Add annotations
ax.annotate('INFLOW\n(fixed velocity)', xy=(0.0, 0.2), xytext=(-0.15, 0.2),
            fontsize=10, ha='right', va='center', color='green', weight='bold',
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

ax.annotate('OUTFLOW\n(open boundary)', xy=(1.6, 0.2), xytext=(1.75, 0.2),
            fontsize=10, ha='left', va='center', color='orange', weight='bold',
            arrowprops=dict(arrowstyle='->', color='orange', lw=2))

ax.annotate('CYLINDER\n(no-slip)', xy=(0.2, 0.2), xytext=(0.2, 0.3),
            fontsize=10, ha='center', va='bottom', color='red', weight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_title('Node Types and Boundary Conditions', fontsize=14, weight='bold')
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Corrected visualization complete!")

In [ ]:
# Visualize dynamics over time - multiple timesteps
import matplotlib.pyplot as plt
import numpy as np


def plot_dynamics_over_time(trajectory, timesteps=[0, 100, 200, 300, 400, 599],
                            field='pressure', title=None):
    """
    Plot a field (pressure, velocity_x, velocity_y, velocity_magnitude) over time.

    Args:
        trajectory: Trajectory dictionary
        timesteps: List of timesteps to visualize
        field: 'pressure', 'velocity_x', 'velocity_y', or 'velocity_magnitude'
    """
    mesh_pos = trajectory['mesh_pos'].numpy()
    node_type = trajectory['node_type'].numpy()

    n_plots = len(timesteps)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    # Prepare data based on field type
    all_values = []
    for t in timesteps:
        if field == 'pressure':
            values = trajectory['pressure'][t].numpy()
        elif field == 'velocity_x':
            values = trajectory['velocity'][t, :, 0].numpy()
        elif field == 'velocity_y':
            values = trajectory['velocity'][t, :, 1].numpy()
        elif field == 'velocity_magnitude':
            vel = trajectory['velocity'][t].numpy()
            values = np.sqrt(vel[:, 0] ** 2 + vel[:, 1] ** 2)
        all_values.append(values)

    # Get global min/max for consistent colorbar
    vmin = min(v.min() for v in all_values)
    vmax = max(v.max() for v in all_values)

    # Plot each timestep
    for idx, (t, values) in enumerate(zip(timesteps, all_values)):
        ax = axes[idx]

        # Plot field values
        scatter = ax.scatter(mesh_pos[:, 0], mesh_pos[:, 1],
                             c=values, s=25, cmap='RdBu_r' if field == 'pressure' else 'viridis',
                             vmin=vmin, vmax=vmax, alpha=0.8)

        # Highlight boundaries
        boundary_mask = node_type != 0
        ax.scatter(mesh_pos[boundary_mask, 0], mesh_pos[boundary_mask, 1],
                   c='black', s=10, alpha=0.3, marker='s')

        ax.set_title(f'Timestep {t}', fontsize=12, weight='bold')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.2)

        # Add colorbar to each subplot
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label(field.replace('_', ' ').title(), fontsize=10)

    fig.suptitle(title or f'{field.replace("_", " ").title()} Evolution Over Time',
                 fontsize=16, weight='bold', y=0.995)
    plt.tight_layout()
    plt.show()


# Visualize pressure dynamics
trajectory = all_trajectories[0]
plot_dynamics_over_time(trajectory, field='pressure',
                        title='Pressure Field Evolution - Cylinder Flow')

In [ ]:
# Visualize velocity X component dynamics
plot_dynamics_over_time(trajectory, field='velocity_x',
                        title='Velocity X Component Evolution - Cylinder Flow')

In [ ]:

# Visualize velocity Y component dynamics
plot_dynamics_over_time(trajectory, field='velocity_y',
                        title='Velocity Y Component Evolution - Cylinder Flow')

In [ ]:

# Visualize velocity magnitude dynamics
plot_dynamics_over_time(trajectory, field='velocity_magnitude',
                        title='Velocity Magnitude Evolution - Cylinder Flow')

In [ ]:

# Create a combined view with all fields at a specific timestep
def plot_all_fields_at_timestep(trajectory, timestep=300):
    """Plot pressure, velocity components, and velocity magnitude at one timestep."""

    mesh_pos = trajectory['mesh_pos'].numpy()
    node_type = trajectory['node_type'].numpy()

    fig, axes = plt.subplots(2, 2, figsize=(16, 14))

    # Prepare data
    pressure = trajectory['pressure'][timestep].numpy()
    vel = trajectory['velocity'][timestep].numpy()
    vel_x = vel[:, 0]
    vel_y = vel[:, 1]
    vel_mag = np.sqrt(vel_x ** 2 + vel_y ** 2)

    fields = [
        (pressure, 'Pressure', 'RdBu_r'),
        (vel_x, 'Velocity X', 'RdBu_r'),
        (vel_y, 'Velocity Y', 'RdBu_r'),
        (vel_mag, 'Velocity Magnitude', 'viridis')
    ]

    axes = axes.flatten()
    for idx, (values, label, cmap) in enumerate(fields):
        ax = axes[idx]

        # Plot field
        scatter = ax.scatter(mesh_pos[:, 0], mesh_pos[:, 1],
                             c=values, s=30, cmap=cmap, alpha=0.8)

        # Highlight boundaries
        boundary_mask = node_type != 0
        ax.scatter(mesh_pos[boundary_mask, 0], mesh_pos[boundary_mask, 1],
                   c='black', s=15, alpha=0.4, marker='s')

        # Add velocity vectors for velocity plots
        if 'Velocity' in label and label != 'Velocity Magnitude':
            step = max(1, len(mesh_pos) // 80)
            ax.quiver(mesh_pos[::step, 0], mesh_pos[::step, 1],
                      vel[::step, 0], vel[::step, 1],
                      alpha=0.4, scale=10, width=0.003, color='white')

        ax.set_title(label, fontsize=13, weight='bold')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.2)

        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label(label, fontsize=10)

    fig.suptitle(f'All Fields at Timestep {timestep}', fontsize=16, weight='bold', y=0.995)
    plt.tight_layout()
    plt.show()


# Visualize all fields at timestep 300 (middle of simulation)
plot_all_fields_at_timestep(trajectory, timestep=300)

In [ ]:
# Animation with ALL timesteps
import matplotlib
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

matplotlib.rcParams['animation.embed_limit'] = 2000  # Set to 100 MB

def create_animation(trajectory, field='pressure', interval=50, skip_frames=1):
    """
    Create an animation of the field evolution.

    Args:
        trajectory: Trajectory dictionary
        field: Field to animate
        interval: Milliseconds between frames
        skip_frames: Show every Nth frame (1 = all frames, 2 = every other frame, etc.)
    """
    mesh_pos = trajectory['mesh_pos'].numpy()
    node_type = trajectory['node_type'].numpy()
    boundary_mask = node_type != 0

    # Prepare all data
    if field == 'pressure':
        all_data = trajectory['pressure'].numpy()
        cmap = 'RdBu_r'
    elif field == 'velocity_magnitude':
        vel = trajectory['velocity'].numpy()
        all_data = np.sqrt(vel[:, :, 0] ** 2 + vel[:, :, 1] ** 2)
        cmap = 'viridis'
    elif field == 'velocity_x':
        all_data = trajectory['velocity'][:, :, 0].numpy()
        cmap = 'RdBu_r'
    elif field == 'velocity_y':
        all_data = trajectory['velocity'][:, :, 1].numpy()
        cmap = 'RdBu_r'

    vmin, vmax = all_data.min(), all_data.max()

    # Create figure
    fig, ax = plt.subplots(figsize=(14, 7))

    # Initial plot
    scatter = ax.scatter(mesh_pos[:, 0], mesh_pos[:, 1],
                         c=all_data[0], s=30, cmap=cmap,
                         vmin=vmin, vmax=vmax, alpha=0.8)
    ax.scatter(mesh_pos[boundary_mask, 0], mesh_pos[boundary_mask, 1],
               c='black', s=15, alpha=0.4, marker='s')

    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    title = ax.set_title(f'{field.replace("_", " ").title()} - Timestep 0',
                         fontsize=14, weight='bold')

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label(field.replace('_', ' ').title(), fontsize=11)

    # Animation update function
    def update(frame):
        scatter.set_array(all_data[frame])
        title.set_text(f'{field.replace("_", " ").title()} - Timestep {frame}')
        return scatter, title

    # Create animation with all timesteps
    frame_indices = range(0, len(all_data), skip_frames)
    anim = FuncAnimation(fig, update, frames=frame_indices,
                         interval=interval, blit=True)

    plt.close()  # Prevent duplicate display

    return anim


# Create animation with ALL 600 timesteps
print("🎬 Creating animation with ALL timesteps...")
print(f"Total timesteps: {trajectory['pressure'].shape[0]}")
print("This may take a moment to generate...")

anim = create_animation(trajectory, field='velocity_magnitude', interval=50, skip_frames=3)

# Display in notebook
HTML(anim.to_jshtml())

In [ ]:
# Time series analysis: Plot values at specific nodes over time
def plot_node_timeseries(trajectory, node_indices=[100, 500, 1000], fields=['pressure', 'velocity_x', 'velocity_y']):
    """Plot time series of field values at specific nodes."""

    fig, axes = plt.subplots(len(fields), 1, figsize=(14, 4 * len(fields)))
    if len(fields) == 1:
        axes = [axes]

    mesh_pos = trajectory['mesh_pos'].numpy()
    timesteps = np.arange(trajectory['pressure'].shape[0])

    for ax_idx, field in enumerate(fields):
        ax = axes[ax_idx]

        for node_idx in node_indices:
            if field == 'pressure':
                values = trajectory['pressure'][:, node_idx].numpy()
                label = f'Node {node_idx} at ({mesh_pos[node_idx, 0]:.3f}, {mesh_pos[node_idx, 1]:.3f})'
            elif field == 'velocity_x':
                values = trajectory['velocity'][:, node_idx, 0].numpy()
                label = f'Node {node_idx} at ({mesh_pos[node_idx, 0]:.3f}, {mesh_pos[node_idx, 1]:.3f})'
            elif field == 'velocity_y':
                values = trajectory['velocity'][:, node_idx, 1].numpy()
                label = f'Node {node_idx} at ({mesh_pos[node_idx, 0]:.3f}, {mesh_pos[node_idx, 1]:.3f})'
            elif field == 'velocity_magnitude':
                vel = trajectory['velocity'][:, node_idx].numpy()
                values = np.sqrt(vel[:, 0] ** 2 + vel[:, 1] ** 2)
                label = f'Node {node_idx} at ({mesh_pos[node_idx, 0]:.3f}, {mesh_pos[node_idx, 1]:.3f})'

            ax.plot(timesteps, values, label=label, linewidth=2, alpha=0.8)

        ax.set_xlabel('Timestep', fontsize=11)
        ax.set_ylabel(field.replace('_', ' ').title(), fontsize=11)
        ax.set_title(f'{field.replace("_", " ").title()} Over Time', fontsize=13, weight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Plot time series for selected nodes
print("📈 Time series at different spatial locations:")
plot_node_timeseries(trajectory, node_indices=[100, 500, 1000, 1500],
                     fields=['pressure', 'velocity_x', 'velocity_y'])

In [ ]:

# Interpolate mesh data to regular grid for matrix visualization
from scipy.interpolate import griddata
import matplotlib
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

matplotlib.rcParams['animation.embed_limit'] = 2000


def create_grid_animation(trajectory, field='pressure', interval=50, skip_frames=1,
                          grid_resolution=(200, 100)):
    """
    Create animation with regular grid (matrix/image) visualization.

    Args:
        trajectory: Trajectory dictionary
        field: Field to animate
        interval: Milliseconds between frames
        skip_frames: Show every Nth frame
        grid_resolution: (width, height) of the regular grid
    """
    mesh_pos = trajectory['mesh_pos'].numpy()
    node_type = trajectory['node_type'].numpy()

    # Prepare all data
    if field == 'pressure':
        all_data = trajectory['pressure'].numpy()
        cmap = 'RdBu_r'
    elif field == 'velocity_magnitude':
        vel = trajectory['velocity'].numpy()
        all_data = np.sqrt(vel[:, :, 0] ** 2 + vel[:, :, 1] ** 2)
        cmap = 'viridis'
    elif field == 'velocity_x':
        all_data = trajectory['velocity'][:, :, 0].numpy()
        cmap = 'RdBu_r'
    elif field == 'velocity_y':
        all_data = trajectory['velocity'][:, :, 1].numpy()
        cmap = 'RdBu_r'

    vmin, vmax = all_data.min(), all_data.max()

    # Create regular grid
    x_min, x_max = mesh_pos[:, 0].min(), mesh_pos[:, 0].max()
    y_min, y_max = mesh_pos[:, 1].min(), mesh_pos[:, 1].max()

    grid_x = np.linspace(x_min, x_max, grid_resolution[0])
    grid_y = np.linspace(y_min, y_max, grid_resolution[1])
    grid_X, grid_Y = np.meshgrid(grid_x, grid_y)

    # Interpolate first frame to get the shape
    print(f"Interpolating to {grid_resolution[0]}x{grid_resolution[1]} grid...")
    first_frame_grid = griddata(mesh_pos, all_data[0], (grid_X, grid_Y), method='cubic')

    # Create obstacle mask (where node_type != 0 near boundaries)
    obstacle_nodes = mesh_pos[node_type == 6]  # Type 6 is obstacle/wall
    if len(obstacle_nodes) > 0:
        # Find obstacle region
        obstacle_grid = griddata(mesh_pos, (node_type == 6).astype(float),
                                 (grid_X, grid_Y), method='nearest')
        obstacle_mask = obstacle_grid > 0.5
    else:
        obstacle_mask = None

    # Create figure
    fig, ax = plt.subplots(figsize=(14, 7))

    # Initial plot with imshow
    im = ax.imshow(first_frame_grid, origin='lower', aspect='auto',
                   extent=[x_min, x_max, y_min, y_max],
                   cmap=cmap, vmin=vmin, vmax=vmax, interpolation='bilinear')

    # Overlay obstacle if exists
    if obstacle_mask is not None:
        obstacle_overlay = np.ma.masked_where(~obstacle_mask, obstacle_mask)
        ax.imshow(obstacle_overlay, origin='lower', aspect='auto',
                  extent=[x_min, x_max, y_min, y_max],
                  cmap='gray', alpha=0.7, vmin=0, vmax=1)

    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    title = ax.set_title(f'{field.replace("_", " ").title()} - Timestep 0',
                         fontsize=14, weight='bold')

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(field.replace('_', ' ').title(), fontsize=11)

    # Pre-compute all interpolated frames for smooth animation
    frame_indices = list(range(0, len(all_data), skip_frames))
    print(f"Pre-computing {len(frame_indices)} interpolated frames...")

    interpolated_frames = []
    for i, frame_idx in enumerate(frame_indices):
        if i % 50 == 0:
            print(f"  Frame {i}/{len(frame_indices)}...")
        grid_data = griddata(mesh_pos, all_data[frame_idx], (grid_X, grid_Y), method='cubic')
        interpolated_frames.append(grid_data)

    print("Creating animation...")

    # Animation update function
    def update(frame_num):
        im.set_array(interpolated_frames[frame_num])
        actual_timestep = frame_indices[frame_num]
        title.set_text(f'{field.replace("_", " ").title()} - Timestep {actual_timestep}')
        return im, title

    # Create animation
    anim = FuncAnimation(fig, update, frames=len(interpolated_frames),
                         interval=interval, blit=True)

    plt.close()  # Prevent duplicate display

    return anim


# Create grid-based animation
print("🎬 Creating matrix/grid animation...")
print(f"Total timesteps: {trajectory['pressure'].shape[0]}")

anim = create_grid_animation(trajectory, field='velocity_magnitude',
                             interval=50, skip_frames=3,
                             grid_resolution=(256, 128))

# Display in notebook
HTML(anim.to_jshtml())